# Démonstration — Agent d'actualité RAG, étape par étape (CRISP-DM)

Ce notebook est un **support de présentation** : on exécute les cellules une à une
(`Maj + Entrée`) et on commente ce qui s'affiche.

Il appelle les fonctions du package `src/agent_rag/` : **le code n'est écrit qu'une fois**,
le notebook ne fait que l'appeler et **montrer** les résultats intermédiaires.

| Étape | Dossier du code | Ce qu'on va montrer |
|---|---|---|
| 1. Métier | `docs/1_comprehension_metier.md` | Le problème et les critères de succès |
| 2. Données | `etape2_donnees/` | Articles bruts, sources, longueurs, défauts |
| 3. Préparation | `etape3_preparation/` | Avant / après nettoyage, découpage en chunks |
| 4. Modélisation | `etape4_modelisation/` | Embeddings, recherche sémantique, **chaîne RAG classique** |
| 5. Évaluation | `etape5_evaluation/` | Hit@k et MRR, comparaison avec une baseline |
| 6. Déploiement | `etape6_deploiement/` | L'**agent** qui décide d'appeler ses outils |

**Pré-requis :** `uv sync`, puis Ollama lancé avec le modèle (`ollama pull qwen2.5:3b`).

In [ ]:
# Configuration commune
import json, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # utile si le package n'est pas installé

import pandas as pd
import matplotlib.pyplot as plt

from agent_rag import config
from agent_rag.commun import llm
from agent_rag.commun.fichiers import ecrire_json

pd.set_option("display.max_colwidth", 80)
print("LLM :", config.MODELE_LLM, "| Embedding :", config.MODELE_EMBEDDING)
print(llm.verifier_disponibilite()[1])

---
# Étape 1 — Compréhension du métier

> On commence par la **question métier**, jamais par le code.

- **Problème :** un LLM seul ne connaît pas l'actualité (date de coupure) et peut inventer.
- **Solution :** un **agent** qui, quand c'est utile, appelle un **outil de recherche** (RAG)
  dans des articles récents, puis répond **en citant ses sources**.
- **Critères de succès** (vérifiés automatiquement à l'étape 5) : voir ci-dessous.

**Démonstration du problème :** posons une question d'actualité au LLM **sans** RAG.

In [ ]:
print("Critères de succès :")
print(f"  Hit@k >= {config.SEUIL_HIT_K} | MRR >= {config.SEUIL_MRR} | "
      f"citation >= {config.SEUIL_CITATION} | fidélité >= {config.SEUIL_FIDELITE}\n")

question_test = "Quelles sont les dernières nouvelles du sport en France ?"
print("LLM SANS RAG :", llm.demander(question_test))
# -> Réponse vague, datée ou inventée : c'est le problème que le RAG va résoudre.

---
# Étape 2 — Compréhension des données

On **collecte** des articles (flux RSS), puis on les **explore** avant toute transformation.

In [ ]:
from agent_rag.etape2_donnees.collecte import collecter
from agent_rag.etape2_donnees.exploration import explorer, afficher_rapport, vers_dataframe

articles_bruts, origine = collecter()
ecrire_json(articles_bruts, config.FICHIER_ARTICLES_BRUTS)   # on ne modifie jamais la source
df_brut = vers_dataframe(articles_bruts)
print(f"{len(articles_bruts)} articles ({origine})")
df_brut.head()

In [ ]:
# Rapport d'exploration : le calcul (explorer) est séparé de l'affichage
stats = explorer(articles_bruts)
afficher_rapport(stats, exemple=articles_bruts[0])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
pd.Series(stats["par_source"]).plot.bar(ax=axes[0], title="Articles par source", rot=0)
df_brut["nb_mots_resume"].plot.hist(ax=axes[1], bins=20, title="Longueur des résumés (mots)")
axes[1].axvline(config.TAILLE_CHUNK, color="red", linestyle="--",
                label=f"taille d'un chunk ({config.TAILLE_CHUNK})")
axes[1].legend(); plt.tight_layout(); plt.show()
# -> Question aux étudiants : faut-il découper ces résumés ? (regardez la ligne rouge)

---
# Étape 3 — Préparation des données

Sélection → Nettoyage → Construction → Intégration → Formatage (chunks).

In [ ]:
from agent_rag.etape3_preparation.nettoyage import nettoyer_texte
from agent_rag.etape3_preparation.decoupage import decouper_en_chunks

sale = "<p>Le prix de l&#39;essence   augmente <b>encore</b>.</p>\n"
print("AVANT :", repr(sale))
print("APRÈS :", repr(nettoyer_texte(sale)))

print()
for i, morceau in enumerate(decouper_en_chunks(" ".join(f"mot{i}" for i in range(1, 16)),
                                               taille=6, chevauchement=2), 1):
    print(f"chunk {i} : {morceau}")
# -> Les 2 derniers mots d'un chunk sont répétés au début du suivant : le chevauchement.

In [ ]:
from agent_rag.etape3_preparation.etape import preparer

chunks, stats_prep = preparer(articles_bruts)
ecrire_json(chunks, config.FICHIER_CHUNKS)
print(stats_prep)
pd.DataFrame(chunks)[["id_chunk", "source", "titre", "texte"]].head()

---
# Étape 4 — Modélisation : le RAG

**R**etrieve (embeddings + index vectoriel) → **A**ugment (consigne + passages) → **G**enerate (LLM).

Un **embedding** transforme un texte en vecteur. Deux textes de **même sens** → vecteurs **proches**.

In [ ]:
from agent_rag.etape4_modelisation.embeddings import vectoriser

phrases = [
    "La hausse des prix inquiète les ménages.",      # référence
    "L'inflation pèse sur le budget des familles.",  # même sens, AUCUN mot commun
    "Les prix du match de football ont augmenté.",   # mots communs, sens différent
    "Le XV de France affronte l'Afrique du Sud.",    # sans rapport
]
v = vectoriser(phrases)
print("Forme :", v.shape, "— 5 premières valeurs du 1er vecteur :", v[0][:5].round(3))
pd.DataFrame({"phrase": phrases, "similarité avec la 1re": (v @ v[0]).round(3)})
# -> La phrase 2 est la plus proche sans partager un seul mot : la recherche SÉMANTIQUE.

In [ ]:
# R — Construire l'index vectoriel, puis chercher
from agent_rag.etape4_modelisation.index_vectoriel import IndexVectoriel

index = IndexVectoriel.construire(chunks)
index.sauvegarder()
print("Index :", index.dimensions)
pd.DataFrame(index.rechercher("match de football", k=5))[["score", "source", "titre"]]

In [ ]:
# R + A + G — La chaîne RAG classique (recherche systématique)
from agent_rag.etape4_modelisation.rag import ChaineRAG

rag = ChaineRAG(index.rechercher)
reponse_rag = rag.repondre(question_test)
print("Passages fournis au LLM :")
for p in reponse_rag.passages:
    print(f"  [{p['score']:.3f}] {p['titre'][:90]}")
print("\nRÉPONSE AVEC RAG :\n" + reponse_rag.texte)
# -> Comparez avec la réponse SANS RAG de l'étape 1.

---
# Étape 5 — Évaluation

1. Le LLM génère une **question** pour quelques articles (jeu de test synthétique).
2. On regarde à quel **rang** la recherche retrouve l'article d'origine.
3. **Hit@1, Hit@k, MRR**, comparés à une **baseline** par mots-clés.

L'évaluation des **réponses** (citation, fidélité, RAG vs agent) est plus lente :
`python main.py evaluation --generation` dans un terminal.

In [ ]:
from agent_rag.etape5_evaluation.etape import obtenir_jeu_de_test
from agent_rag.etape5_evaluation.eval_recherche import evaluer_recherche
from agent_rag.etape5_evaluation.baseline import recherche_par_mots_cles

jeu_de_test = obtenir_jeu_de_test(index)
k = config.NB_RESULTATS
scores = pd.DataFrame({
    "Baseline mots-clés": evaluer_recherche(lambda q, k: recherche_par_mots_cles(q, index.chunks, k), jeu_de_test, k),
    "RAG sémantique": evaluer_recherche(index.rechercher, jeu_de_test, k),
})
scores

In [ ]:
scores.plot.bar(rot=0, ylim=(0, 1.05), title="Qualité de la recherche")
plt.axhline(config.SEUIL_HIT_K, color="gray", linestyle="--", linewidth=1)
plt.show()
# -> Objectifs non atteints ? Retour aux étapes 3-4 : CRISP-DM est un CYCLE.

---
# Étape 6 — Déploiement : l'agent et ses outils

L'agent **décide** s'il doit chercher, **quoi** chercher, et peut chercher **plusieurs fois**.
On regarde la **boucle agentique** : appel d'outil → exécution par notre code → réponse.

`[outil (recherche de secours)]` = le petit LLM voulait répondre sans chercher : le **garde-fou**
du code a cherché à sa place (voir `docs/ARCHITECTURE.md`, section 7).

In [ ]:
from agent_rag.etape6_deploiement.agent import Agent
from agent_rag.etape6_deploiement.outils import creer_outils

outils = creer_outils(index)
agent = Agent(outils)

# Ce que le LLM « voit » d'un outil : uniquement sa description
print(json.dumps(outils.schemas()[0], ensure_ascii=False, indent=2))

In [ ]:
def poser(question):
    reponse = agent.repondre(question)
    for e in reponse.etapes:
        origine = " (recherche de secours)" if e.automatique else ""
        print(f"   [outil{origine}] {e.nom}({json.dumps(e.arguments, ensure_ascii=False)})")
    print("\nAGENT :", reponse.texte)
    return reponse

poser("Bonjour, que sais-tu faire ?")      # -> aucun outil : l'agent décide seul

In [ ]:
r = poser(question_test)                   # -> appel de rechercher_actualites
print("\nSources consultées :", [s["titre"][:60] for s in r.sources])

In [ ]:
poser("Et en politique ?")                 # -> question de suivi : la mémoire de la conversation

In [ ]:
# Dans les coulisses : tous les messages échangés
for m in agent.historique:
    if m.get("tool_calls"):
        contenu = "APPEL D'OUTIL -> " + ", ".join(
            f"{a['function']['name']}({a['function']['arguments']})" for a in m["tool_calls"])
    else:
        contenu = (m.get("content") or "")[:150].replace("\n", " ")
    print(f"[{m['role']:>9}] {contenu}")

---
## Pour aller plus loin

- Discuter librement avec l'agent : `python main.py agent`
- L'API REST : `python main.py api` puis http://localhost:8000/docs
- L'interface web : `python main.py interface`
- Exercices : section *Pistes d'exercices* du `README.md` ; cahier des charges : `docs/GUIDE_ETUDIANTS.md`